# GENIE systematics inspection

Load precomputed GENIE `cov_mat_dict.pkl` products (from `run_syst_genie_chunked.sh` /
`syst_genie_aggregate`) and make **rate** + **xsec** uncertainty plots.

**Sections**
1. **Per-mode knobs** — pick a mode (QE, MEC, …); show each knob’s contribution
2. **Per-mode totals** — mode contributions + total GENIE; frac. cov + corr (side-by-side) per mode  
   (Ar23p as its own mode, and Ar23p knobs redistributed; “distributed” only in save names)
3. **Top knobs + total** — total GENIE + the 10 largest *integrated* families (binned knobs collapsed);
   frac. cov + corr for the total; contribution % → CSV

Helpers live in `analysis_village.numucc_1p0pi.syst_genie_inspect`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE
from analysis_village.numucc_1p0pi.final_selected_evt_vars import CORE_SELECTED_EVT_VARIABLE_CONFIGS
from analysis_village.numucc_1p0pi.syst_genie_inspect import (
    DISTRIBUTED_MODE_ORDER,
    MODE_ORDER,
    PHYSICS_MODES,
    all_knob_parts,
    assign_ar23p_knob_to_mode,
    assign_other_mode_knob_to_bucket,
    contribution_rows,
    display_mode_name,
    integrated_frac_unc_pct,
    knob_parts_for_mode,
    load_groups,
    mode_totals_ar23p_distributed,
    mode_totals_ar23p_standalone,
    plot_frac_unc_breakdown,
    plot_mode_frac_unc,
    show_cov_corr_heatmaps,
    sum_cov_fracs,
    top_n_knobs_by_integrated,
    write_contribution_csv,
)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
SYST_BASE = Path("/exp/sbnd/data/users/munjung/xsec/numucc_1p0pi")
OUT_DIR = Path(PLOTS_BASE) / "genie_syst_inspect"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Prefer newest Ar23p disk when present; other modes: Aug24 sel_mup campaign.
_AR23P_CANDIDATES = [
    SYST_BASE / "syst_disk_sel_mup_20260912_Ar23p/GENIE/cov_mat_dict.pkl",
    SYST_BASE / "syst_disk_sel_mup_Aug24_Ar23p/GENIE/cov_mat_dict.pkl",
]
_ar23p = next((p for p in _AR23P_CANDIDATES if p.is_file()), _AR23P_CANDIDATES[0])

GROUP_SOURCES = {
    "CCQE": SYST_BASE / "syst_disk_sel_mup_Aug24_CCQE/GENIE/cov_mat_dict.pkl",
    "MEC": SYST_BASE / "syst_disk_sel_mup_Aug24_MEC/GENIE/cov_mat_dict.pkl",
    "RES": SYST_BASE / "syst_disk_sel_mup_Aug24_RES/GENIE/cov_mat_dict.pkl",
    "nonRES": SYST_BASE / "syst_disk_sel_mup_Aug24_nonRES/GENIE/cov_mat_dict.pkl",
    "DIS": SYST_BASE / "syst_disk_sel_mup_Aug24_DIS/GENIE/cov_mat_dict.pkl",
    "Other": SYST_BASE / "syst_disk_sel_mup_Aug24_Other/GENIE/cov_mat_dict.pkl",
    "Ar23p": _ar23p,
}

# Override paths via env, e.g. GENIE_INSPECT_CCQE=/path/to/cov_mat_dict.pkl
for mode in list(GROUP_SOURCES):
    env = os.environ.get(f"GENIE_INSPECT_{mode}")
    if env:
        GROUP_SOURCES[mode] = Path(env)

VARS = ["integrated", "tki-del_Tp", "tki-del_alpha", "tki-del_phi"]
COV_TYPES = ["rate", "xsec"]
SAVE_FIGS = True
FIG_DPI = 140
TOP_N_KNOBS = 10  # section 3
INSPECT_MODE = os.environ.get("GENIE_INSPECT_MODE", "CCQE")  # section 1

vc_by = {vc.var_save_name: vc for vc in CORE_SELECTED_EVT_VARIABLE_CONFIGS}
print("OUT_DIR =", OUT_DIR)
print("INSPECT_MODE =", INSPECT_MODE)
for m, p in GROUP_SOURCES.items():
    print(f"  {m:7s} exists={Path(p).is_file()}  {p}")


In [ ]:
# Load all available modes (skip missing paths with a warning)
_sources = {m: p for m, p in GROUP_SOURCES.items() if Path(p).is_file()}
_missing = [m for m in GROUP_SOURCES if m not in _sources]
if _missing:
    print("WARNING: missing modes (skipped):", _missing)
if not _sources:
    raise FileNotFoundError("No GROUP_SOURCES files found — edit paths in the config cell")

groups = load_groups(_sources)

# Variables present in every loaded mode
common_vars = set.intersection(*(set(g.keys()) for g in groups.values()))
plot_vars = [v for v in VARS if v in common_vars]
if not plot_vars:
    plot_vars = sorted(common_vars)
print("plot_vars:", plot_vars)


## 1. Per-mode knob contributions

Set `INSPECT_MODE` in the config cell (or `GENIE_INSPECT_MODE`). For each analysis
variable, show **rate** and **xsec** fractional uncertainties from every knob in
that mode, plus the mode total.


In [ ]:
mode = INSPECT_MODE
if mode not in groups:
    raise KeyError(f"INSPECT_MODE={mode!r} not loaded; have {sorted(groups)}")

for slug in plot_vars:
    vc = vc_by.get(slug)
    for kind in ("xsec", "rate"):
        parts, total = knob_parts_for_mode(groups, mode, slug, kind)
        fig, ax = plt.subplots(figsize=(6.4, 4.8))
        plot_frac_unc_breakdown(
            ax, parts, total, vc=vc, kind=kind, top_n=None,
        )
        fig.tight_layout()
        if SAVE_FIGS:
            out = OUT_DIR / f"sec1_{mode}__{slug}__{kind}__knobs.png"
            fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
            print("wrote", out)
        plt.show()


## 2. Per-mode totals (+ Ar23p treatments)

For each variable × (rate, xsec):

* uncertainty curves for each mode and the GENIE total
* fractional covariance + correlation as one wide figure (subplots) **per mode**

**Two Ar23p treatments** (tag only appears in save names, not plot titles)

1. **Standalone** — Ar23p is its own mode  
2. **Distributed**
   - Ar23p: `QE`/`ZExp` → QE; `MEC` → MEC; else → **FSI**
   - Other mode: `*COH` / `*NCEL` → **Other**; remaining Other knobs → **FSI**


In [ ]:
def _section2(distribute_ar23p: bool, tag: str):
    mode_order = DISTRIBUTED_MODE_ORDER if distribute_ar23p else MODE_ORDER
    totals_fn = mode_totals_ar23p_distributed if distribute_ar23p else mode_totals_ar23p_standalone

    for kind in COV_TYPES:
        # --- mode uncertainty (one figure per variable) ---
        for slug in plot_vars:
            mt = totals_fn(groups, slug, kind)
            fig, ax = plt.subplots(figsize=(6.4, 4.8))
            plot_mode_frac_unc(
                ax, mt, vc=vc_by.get(slug), kind=kind, mode_order=mode_order,
            )
            fig.tight_layout()
            if SAVE_FIGS:
                out = OUT_DIR / f"sec2_mode_totals__{tag}__{kind}__{slug}.png"
                fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
                print("wrote", out)
            plt.show()

        # --- per-mode frac cov + corr (one wide figure) ---
        for slug in plot_vars:
            vc = vc_by.get(slug)
            if vc is None:
                print(f"skip heatmaps {slug}: no VariableConfig")
                continue
            mt = totals_fn(groups, slug, kind)
            for mode in mode_order:
                if mode not in mt:
                    continue
                save = None
                if SAVE_FIGS:
                    save = OUT_DIR / f"sec2_covcorr__{tag}__{kind}__{slug}__{mode}.png"
                show_cov_corr_heatmaps(
                    mt[mode], vc,
                    kind=kind,
                    suptitle=display_mode_name(mode),
                    save_path=save,
                    dpi=FIG_DPI,
                )

print("Distributed assignment examples:")
if "Other" in groups and "integrated" in groups["Other"]:
    from analysis_village.numucc_1p0pi.syst_genie_inspect import iter_knob_cov_fracs
    seen = {}
    for kn, _ in iter_knob_cov_fracs(groups["Other"]["integrated"], "xsec"):
        seen.setdefault(assign_other_mode_knob_to_bucket(kn), []).append(kn)
    for dest, kns in sorted(seen.items()):
        print(f"  Other→{dest}: {len(kns)} knobs (e.g. {kns[0]})")
if "Ar23p" in groups and "integrated" in groups["Ar23p"]:
    from analysis_village.numucc_1p0pi.syst_genie_inspect import iter_knob_cov_fracs
    seen = {}
    for kn, _ in iter_knob_cov_fracs(groups["Ar23p"]["integrated"], "xsec"):
        seen.setdefault(assign_ar23p_knob_to_mode(kn), []).append(kn)
    for dest, kns in sorted(seen.items()):
        print(f"  Ar23p→{display_mode_name(dest)}: {len(kns)} knobs (e.g. {kns[0]})")

_section2(distribute_ar23p=False, tag="standalone")
_section2(distribute_ar23p=True, tag="distributed")


## 3. Total GENIE + top 10 knobs

Ranking uses each knob’s fractional uncertainty on **`integrated`** (not an
average over bins of the plotted variable). Binned families (``ZExp_b*``, dials,
``q0bin*``, …) are collapsed into one knob each before ranking.

Plots show the total and those top 10 knobs for every analysis variable ×
(rate, xsec), plus frac. cov + corr for the **total**. Contribution percentages
are written to CSV next to the plots (not drawn on the legend). Legend shows
knob names only (no mode prefix).


In [ ]:
for kind in COV_TYPES:
    # Rank on integrated (binned knobs collapsed into families)
    int_parts = all_knob_parts(
        groups, "integrated", kind, distribute_ar23p=False, collapse_binned=True,
    )
    top_keys = top_n_knobs_by_integrated(int_parts, int_parts, n=TOP_N_KNOBS)
    print(f"[{kind}] top {TOP_N_KNOBS} knobs by integrated frac. unc. (families):")
    for i, k in enumerate(top_keys, 1):
        print(f"  {i:2d}. {k:50s}  {integrated_frac_unc_pct(int_parts[k]):6.3f}%")

    for slug in plot_vars:
        vc = vc_by.get(slug)
        parts = all_knob_parts(
            groups, slug, kind, distribute_ar23p=False, collapse_binned=True,
        )
        mt = mode_totals_ar23p_standalone(groups, slug, kind)
        total = sum_cov_fracs(list(mt.values()))
        show_parts = {k: parts[k] for k in top_keys if k in parts}
        use_int_score = (slug == "integrated") or (vc is not None and getattr(vc, "var_save_name", "") == "integrated")

        fig, ax = plt.subplots(figsize=(6.4, 4.8))
        plot_frac_unc_breakdown(
            ax, show_parts, total, vc=vc,
            kind=kind,
            ordered_keys=top_keys,
            show_pct_in_legend=False,
            legend_knob_only=True,
        )
        fig.tight_layout()
        if SAVE_FIGS:
            out = OUT_DIR / f"sec3_top{TOP_N_KNOBS}__{kind}__{slug}.png"
            fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
            print("wrote", out)
            csv_rows = contribution_rows(
                show_parts, total, kind=kind, slug=slug,
                ordered_keys=top_keys, use_integrated_score=use_int_score,
            )
            csv_path = OUT_DIR / f"sec3_top{TOP_N_KNOBS}__{kind}__{slug}__contrib.csv"
            write_contribution_csv(csv_path, csv_rows)
            print("wrote", csv_path)
        plt.show()

        if total is not None and vc is not None:
            save = None
            if SAVE_FIGS:
                save = OUT_DIR / f"sec3_covcorr__{kind}__{slug}__total.png"
            show_cov_corr_heatmaps(
                total, vc, kind=kind, suptitle="GENIE",
                save_path=save, dpi=FIG_DPI,
            )
